<a href="https://colab.research.google.com/github/soleildayana/Oumuamua-Project/blob/main/Orbit_OUMUAMUA_Viewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%pip install pymcel -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.9 MB/s eta 0:00:00


In [8]:
import numpy as np
import plotly.graph_objects as go
from dataclasses import dataclass
import pymcel as pc
import spiceypy as spy
import matplotlib.pyplot as plt

Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!


# Visualizador de órbitas 3D aplicando las 3 rotaciones de Euler con elementos orbitales

## Rutina para aplicar las rotaciones

La siguiente función `orbital_elements_to_orbit_3d` toma los elementos orbitales clásicos (excentricidad, semieje mayor, inclinación, longitud del nodo ascendente y argumento del periapsis) y genera las coordenadas 3D de la órbita en el sistema de referencia astronómico. Utiliza las tres rotaciones de Euler para orientar la órbita correctamente en el espacio.


In [9]:
def orbital_elements_to_orbit_3d(e, a, i, node, peri, num_points=100):
    """
    Convierte los elementos orbitales clásicos en una órbita 3D en el espacio.

    Args:
        e (float): Excentricidad.
        a (float): Semieje mayor.
        i (float): Inclinación (en radianes).
        node (float): Longitud del nodo ascendente (en radianes).
        peri (float): Argumento del periapsis (en radianes).
        num_points (int): Número de puntos para generar la órbita.

    Returns:
        np.ndarray: Array de Nx3 con las coordenadas (x, y, z) de la órbita.
    """

    # Calcular el parámetro semilato recto (p)
    p = a * (1 - e**2) if e < 1 else a * (e**2 - 1) # Asegurar que 'p' es positivo para hiperbolas
    if e == 1: # Caso parabolico
        p = 2 * a # a es la distancia al periapsis para una parabola

    # Generar ángulos de anomalía verdadera (f) para dibujar la órbita
    # Si es una órbita hiperbólica, los límites de f son diferentes.
    if e < 1: # Elipse
        fs = np.linspace(0, 2 * np.pi, num_points)
    elif e == 1: # Parábola, se necesita un rango adecuado
        # Para una parábola, la anomalía verdadera puede ir de -pi a pi, pero la órbita es infinita.
        # Usaremos un rango que muestre una buena parte de la trayectoria.
        f_limit = np.arccos(-1/e + 0.001) # Límite para evitar division por cero
        fs = np.linspace(-f_limit, f_limit, num_points) # Valores cercanos al periapsis
    else: # Hipérbola
        # Los límites de la anomalía verdadera para una hipérbola
        f_limit = np.arccos(-1/e + 0.001) # Pequeña perturbación para evitar arccos(-1)
        fs = np.linspace(-f_limit + 0.1, f_limit - 0.1, num_points) # Evitar los asíntotas

    # Calcular la distancia radial (r) para cada f
    rs = p / (1 + e * np.cos(fs))

    # Coordenadas en el plano perifocal (xy) con z=0
    xfs = rs * np.cos(fs)
    yfs = rs * np.sin(fs)
    zfs = np.zeros_like(xfs)

    # Convertir a array de 3xN para multiplicación matricial
    r_perifocal = np.array([xfs, yfs, zfs])

    # Matrices de rotación (del sistema perifocal al astronómico)
    # Nota: spiceypy.rotate(angle, axis) devuelve la matriz de rotación CONTRARIA
    # a la que se usa para transformar un vector, si se usa como M @ v.
    # Para pasar del sistema perifocal al astronómico, las rotaciones son las inversas.
    # O lo que es lo mismo, rotar por ángulos negativos en el orden inverso.

    # Rotaciones de Euler (Z-X-Z)
    # 1. Rotación alrededor del eje Z por -Omega (Longitud del Nodo Ascendente)
    Rz_minus_Omega = spy.rotate(-node, 3)
    # 2. Rotación alrededor del eje X por -i (Inclinación)
    Rx_minus_i = spy.rotate(-i, 1)
    # 3. Rotación alrededor del eje Z por -w (Argumento del Periapsis)
    Rz_minus_peri = spy.rotate(-peri, 3)

    # Combinar las matrices para obtener la transformación de perifocal a astronómico
    M_perifocal2astro = Rz_minus_Omega @ Rx_minus_i @ Rz_minus_peri

    # Aplicar la transformación
    r_astro = M_perifocal2astro @ r_perifocal

    # Transponer para obtener un array de Nx3 (N puntos, 3 coordenadas)
    return r_astro.T

## Rutina para generar el visualizador

In [21]:
def create_asteroid_viewer(a, e, i, node, peri, name="Asteroide", grid_lim=2.5):
    """
    Genera y muestra un visualizador 3D para un cuerpo menor.
    """
    # Elementos orbitales de la Tierra (J2000)
    deg = np.pi/180
    a_earth = 1.0000010178
    e_earth = 0.0167086
    i_earth = 0.00005 * deg
    node_earth = -11.26064 * deg
    peri_earth = 102.94719 * deg

    # 1. Generar trayectorias
    orbit_body = orbital_elements_to_orbit_3d(e=e, a=a, i=i, node=node, peri=peri, num_points=600)
    orbit_earth = orbital_elements_to_orbit_3d(e=e_earth, a=a_earth, i=i_earth, node=node_earth, peri=peri_earth, num_points=400)

    fig = go.Figure()

    # --- Sol ---
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode='markers',
                               marker=dict(size=8, color='yellow', symbol='diamond'), name='Sol'))

    # --- Dirección Vernal (X) ---
    fig.add_trace(go.Scatter3d(x=[0, grid_lim*0.5], y=[0, 0], z=[0, 0],
                               mode='lines', line=dict(color='magenta', width=2), name='Eje X (Vernal)'))

    # --- Órbita de la Tierra ---
    fig.add_trace(go.Scatter3d(x=orbit_earth[:, 0], y=orbit_earth[:, 1], z=orbit_earth[:, 2],
                               mode='lines', line=dict(color='#40a0ff', width=2, dash='dot'), name='Órbita Tierra'))

    # --- Órbita del Objeto ---
    fig.add_trace(go.Scatter3d(x=orbit_body[:, 0], y=orbit_body[:, 1], z=orbit_body[:, 2],
                               mode='lines', line=dict(color='white', width=4), name=f'Órbita {name}'))

    # --- Picket-fence (Líneas de inclinación) ---
    for p in orbit_body[::5]:
        fig.add_trace(go.Scatter3d(
            x=[p[0], p[0]], y=[p[1], p[1]], z=[0, p[2]],
            mode='lines',
            line=dict(color='rgba(200, 200, 200, 0.4)', width=2),
            showlegend=False
        ))

    # --- Periapsis Marker ---
    peri_pos = orbit_body[0] if e < 1 else orbit_body[len(orbit_body)//2] # Aproximación visual
    fig.add_trace(go.Scatter3d(x=[orbit_body[0,0]], y=[orbit_body[0,1]], z=[orbit_body[0,2]],
                               mode='markers+text', text=['Peri'], textposition='top center',
                               marker=dict(color='red', size=5), name='Periapsis'))

    # --- Línea de Nodos ---
    node_vec = np.array([np.cos(node), np.sin(node), 0]) * (grid_lim * 0.7)
    fig.add_trace(go.Scatter3d(
        x=[-node_vec[0], node_vec[0]], y=[-node_vec[1], node_vec[1]], z=[0, 0],
        mode='lines', line=dict(color='#00ffff', width=3, dash='longdash'), name='Línea de Nodos'
    ))

    # --- Plano Eclíptico ---
    grid_r = np.linspace(-grid_lim, grid_lim, 20)
    xx, yy = np.meshgrid(grid_r, grid_r)
    fig.add_trace(go.Surface(x=xx, y=yy, z=np.zeros_like(xx),
                             opacity=0.1, showscale=False, colorscale=[[0, 'cyan'], [1, 'cyan']], hoverinfo='skip'))

    fig.update_layout(
        title=f"Simulación de Órbita: {name}",
        scene=dict(
            xaxis=dict(title='X (AU)', range=[-grid_lim, grid_lim]),
            yaxis=dict(title='Y (AU)', range=[-grid_lim, grid_lim]),
            zaxis=dict(title='Z (AU)', range=[-grid_lim, grid_lim]),
            aspectmode='cube',
            bgcolor='black',
            camera=dict(eye=dict(x=1.8, y=1.2, z=1.2))
        ),
        template="plotly_dark",
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()

## Ejemplo de prueba: Apophis

In [24]:
# Elementos orbitales de 'Oumuamua (1I/2017 U1)
deg_to_rad = np.pi / 180

e = 1.201133796102373
a = -1.27234500742808 # au
q = 0.2559115812959116 # au
i = 122.7417062847286 * deg_to_rad
node = 24.59690955523242 * deg_to_rad
peri = 241.8105360304898 * deg_to_rad

print(f"Elementos orbitales de 'Oumuamua:")
print(f"  Semieje mayor (a):     {a:.6f} AU")
print(f"  Excentricidad (e):     {e:.6f}")
print(f"  Inclinación (i):       {i/deg_to_rad:.2f} grados")
print(f"  Nodo Ascendente:       {node/deg_to_rad:.2f} grados")
print(f"  Arg. Periapsis:        {peri/deg_to_rad:.2f} grados")

Elementos orbitales de 'Oumuamua:
  Semieje mayor (a):     -1.272345 AU
  Excentricidad (e):     1.201134
  Inclinación (i):       122.74 grados
  Nodo Ascendente:       24.60 grados
  Arg. Periapsis:        241.81 grados


In [25]:
create_asteroid_viewer(
    a=a,
    e=e,
    i=i,
    node=node,
    peri=peri,
    name="oumuamua"
)